In [2]:
# neuron attribution


def get_llm_block(llm, llm_name):
    if llm_name == "gpt2":
        block = llm.transformer.h
    elif 'meta-llama' in llm_name:
        block = llm.model.layers
    elif 'Qwen' in llm_name:
        block = llm.model.layers
    else:
        raise ValueError(f"Unsupported model: {llm_name}")
    return block


def get_num_layers(llm_name):
    if llm_name == "meta-llama/Meta-Llama-3.1-8B-Instruct":
        return 32
    elif llm_name == "Qwen/Qwen2.5-7B-Instruct":
        return 28
    else:
        raise ValueError(f"Unsupported model: {llm_name}")

def get_mlp_down_proj(llm_name, block):
    if 'meta-llama' in llm_name:
        module = block.mlp.down_proj
    elif 'Qwen' in llm_name:
        module = block.mlp.down_proj
    else:
        raise ValueError(f"Unsupported model: {llm_name}")
    return module

def get_mlp_up_proj(llm_name, block):
    if 'meta-llama' in llm_name:
        module = block.mlp.up_proj
    elif 'Qwen' in llm_name:
        module = block.mlp.up_proj
    else:
        raise ValueError(f"Unsupported model: {llm_name}")
    return module

import torch 
class KeyHiddenStateHook:
        def __init__(self):
            self.hidden_states = None
            self.probe_positions = None
            
        def __call__(self, module, input, output):
            # Store hidden states on CPU immediately and convert to float16
            self.hidden_states = []
            for i  in range(self.probe_positions.shape[0]):
                probe_position = self.probe_positions[i]
                self.hidden_states.append(input[0][i, probe_position, :].clone().detach().cpu().half())
            self.hidden_states = torch.stack(self.hidden_states)
            
        def clear(self):
            self.hidden_states = None

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model = AutoModelForCausalLM.from_pretrained("meta-llama/Meta-Llama-3-8B-Instruct")
tokenizer = AutoTokenizer.from_pretrained("meta-llama/Meta-Llama-3-8B-Instruct")

input = "안녕하세요"

model.forward(tokenizer.encode(input, return_tensors="pt"))

In [ ]:
postive_samples = []
negative_samples = [] 
from torch.utils.data import Dataset, DataLoader

class SparseProberDataset(Dataset):
    def __init__(self, postive_samples_text, negative_samples_text):
        self.postive_samples_text = postive_samples_text
        self.negative_samples_text = negative_samples_text

    def __len__(self):
        return len(self.postive_samples) + len(self.negative_samples)
    
    

In [ ]:
model_name = "meta-llama/Meta-Llama-3-8B-Instruct"
blocks = get_llm_block(model, model_name)
layer_indices = list(range(len(blocks)))  # [::4] + [len(blocks) - 1] # every 4 layers and the last layer
    
hooks = []
for layer_idx in layer_indices:
    key_hook = KeyHiddenStateHook()
    key_activation = get_mlp_down_proj(model_name, blocks[layer_idx])
    handle_key = key_activation.register_forward_hook(key_hook)
    hooks.append(key_hook)
    
    layer_key_hidden_states = {layer: [] for layer in args.layer_indices}
    layer_value_hidden_states = {layer: [] for layer in args.layer_indices}
    all_labels = []
        for batch in pbar:
            input_ids = batch['input_ids'].to(model.device)
            attention_mask = batch['attention_mask'].to(model.device)
            sample_ids = batch['sample_ids']
            probe_positions = batch['probe_positions']
            labels = batch['labels']
            for key_hook in key_hooks:
                key_hook.probe_positions = probe_positions
            for value_hook in value_hooks:
                value_hook.probe_positions = probe_positions
            
            with torch.no_grad():
                generated_ids_batch = model.forward(input_ids, 
                                                    attention_mask=attention_mask,)  
                torch.cuda.empty_cache()  
                input_ids = input_ids.cpu()
                attention_mask = attention_mask.cpu()
                
            labels = labels.unsqueeze(1)
            expanded_labels = labels.repeat_interleave(probe_positions.shape[1]).reshape(labels.shape[0], -1)
            all_labels.append(expanded_labels)
            for index, sample_id in enumerate(sample_ids):
                sample_id = sample_id.item()
                for idx, layer_idx in enumerate(args.layer_indices):
                    key_hidden_states = key_hooks[idx].hidden_states  # Shape: [batch_size, sequence_length, hidden_size]
                    value_hidden_states = value_hooks[idx].hidden_states  # Shape: [batch_size, sequence_length, hidden_size]
                    last_token_key_hidden_state = key_hidden_states[index,:,:]  # Shape: [hidden_size]
                    last_token_value_hidden_state = value_hidden_states[index,:,:]  # Shape: [hidden_size]
                    layer_key_hidden_states[layer_idx].append(last_token_key_hidden_state)
                    layer_value_hidden_states[layer_idx].append(last_token_value_hidden_state)